# Figures for *Introduction to Deep Learning for Chemistry*This notebook regenerates **every figure and every number** used in`lecture_04-intro-deep-learning.pptx`.Run it top to bottom. It will:1. build the molecular parity dataset (counts of H, C, N, O → is the electron count odd?),2. train a 4 → 5 → 1 neural network on it, in plain NumPy,3. compute the data behind each figure and preview it with matplotlib,4. write **`figure_data.json`**, which `build_deck.py` reads to draw the figures in   PowerPoint as *native, editable shapes and charts*.Only `numpy` and `matplotlib` are required.> **Why the JSON step?** The slides do not contain images. Every plot is a real PowerPoint> chart and every schematic is a real PowerPoint shape, so you can click and edit them.> This notebook produces the numbers; `build_deck.py` draws them.

In [ ]:
import json, mathimport numpy as npimport matplotlib.pyplot as plt# House style, matched to the Caltech / bootcamp deckPEACH, ORANGE, ORANGE_D = "#FBE4D6", "#E97132", "#B4551F"TEAL, TEAL_L            = "#156082", "#4E9AB8"PURPLE, INK, GRAY, LGRAY = "#A02B93", "#20242E", "#6E7480", "#D9DCE3"plt.rcParams.update({    "font.size": 11, "axes.edgecolor": GRAY, "axes.labelcolor": INK,    "text.color": INK, "xtick.color": GRAY, "ytick.color": GRAY,    "axes.spines.top": False, "axes.spines.right": False,    "figure.facecolor": "white", "legend.frameon": False,})DATA = {}          # everything the deck builder needs ends up in hererng = np.random.default_rng(7)print("ready")

---## 1. The chemistry taskInputs are the atom counts $n_H, n_C, n_N, n_O$. The target is the **parity of the totalelectron count** — odd means an unpaired electron, i.e. a radical.$$\text{electrons} = 1\,n_H + 6\,n_C + 7\,n_N + 8\,n_O$$Because 6 and 8 are even, only hydrogen and nitrogen can flip the parity:$$\text{parity} = (n_H + n_N) \bmod 2$$That is an **XOR**, which is exactly the function a linear model cannot represent.This single fact is the spine of the whole lecture.

In [ ]:
Z = {"H": 1, "C": 6, "N": 7, "O": 8}ZVEC = np.array([Z["H"], Z["C"], Z["N"], Z["O"]])MAXC = 3            # molecules with up to 3 of each elementdef make_parity_data(n=1500, maxc=MAXC, seed=1):    r = np.random.default_rng(seed)    counts = r.integers(0, maxc + 1, size=(n, 4))    counts = counts[counts.sum(1) > 0]          # drop the empty molecule    electrons = counts @ ZVEC    y = (electrons % 2).astype(float)           # 1 = odd, 0 = even    return counts.astype(float), yXtr, ytr = make_parity_data(1500, seed=1)Xte, yte = make_parity_data(700,  seed=99)print(f"train {len(Xtr)} molecules, test {len(Xte)}")print("fraction odd:", ytr.mean().round(3))# sanity check: parity really is (nH + nN) mod 2assert np.all(((Xtr[:, 0] + Xtr[:, 2]) % 2) == ytr)print("verified: parity == (nH + nN) mod 2")

In [ ]:
# The example molecules shown in the table on the slideTABLE_MOLS = [("CH4", [4,1,0,0]), ("H2O", [2,0,0,1]), ("NH3", [3,0,1,0]),              ("NO",  [0,0,1,1]), ("CH3", [3,1,0,0]), ("C6H6", [6,6,0,0])]rows = []for name, c in TABLE_MOLS:    e = int(np.dot(c, ZVEC))    rows.append({"name": name, "counts": c, "electrons": e,                 "parity": "odd" if e % 2 else "even"})DATA["parity_table"] = rowsfor r_ in rows:    print(f"{r_['name']:6s} {r_['counts']}  e={r_['electrons']:3d}  {r_['parity']}")

---## 2. A linear model, and how its weights are trainedFirst the easy case: one feature, one weight, one bias. The loss is a clean parabola in$w$ — a single minimum we can even solve for in closed form. This is the baseline ofintuition that neural networks will then complicate.

In [ ]:
n = 24x_lin = np.linspace(0.2, 4.6, n)y_lin = 1.35 * x_lin + 0.6 + rng.normal(0, 0.45, n)A = np.vstack([x_lin, np.ones_like(x_lin)]).Tw_fit, b_fit = np.linalg.lstsq(A, y_lin, rcond=None)[0]def lin_loss(w, b):    return np.mean((y_lin - (w*x_lin + b))**2)def lin_grad(w, b):    err = (w*x_lin + b) - y_lin    dw = np.mean(2*err*x_lin)    db = np.mean(2*err)    return np.array([dw, db])# gradient descent from a deliberately-off (w, b), animated alongside the loss# surface it is descending -- it lands right on the closed-form lstsq solutionw0, b0, lr, n_steps = -0.4, -1.6, 0.1, 80p = np.array([w0, b0]); path = [p.copy()]for _ in range(n_steps):    p = p - lr*lin_grad(*p)    path.append(p.copy())path = np.array(path)ws = np.linspace(-0.6, 3.4, 60)loss_curve = [float(np.mean((y_lin - (w * x_lin + b_fit))**2)) for w in ws]DATA["linear_fit"] = {    "x": x_lin.round(4).tolist(), "y": y_lin.round(4).tolist(),    "w": float(round(w_fit, 4)), "b": float(round(b_fit, 4)),    "ws": ws.round(4).tolist(), "loss": [round(v, 4) for v in loss_curve],    "path_w": path[:, 0].round(4).tolist(), "path_b": path[:, 1].round(4).tolist(),}# ---- animation: fit line (left) in lockstep with its position on the full# 2-D (w, b) loss surface (right) ----wg, bg = np.meshgrid(np.linspace(-1.0, 2.2, 140), np.linspace(-2.2, 1.4, 140))Zg = np.array([[lin_loss(a, b) for a in wg[0]] for b in bg[:, 0]])xs = np.linspace(0, 5, 50)from matplotlib.animation import FuncAnimation, PillowWriterfrom IPython.display import Image, displayfig, (axL, axR) = plt.subplots(1, 2, figsize=(9.5, 3.6))axL.scatter(x_lin, y_lin, color=TEAL, s=30, zorder=3)fitline, = axL.plot([], [], color=ORANGE, lw=2.5)title_l = axL.set_title("", loc="left")axL.set_xlabel("feature x"); axL.set_ylabel("target y")axL.set_xlim(0, 5); axL.set_ylim(y_lin.min() - 1, y_lin.max() + 1)axR.contourf(wg, bg, Zg, levels=np.linspace(0, 8, 25), cmap="Blues_r", extend="max")pathline, = axR.plot([], [], color=ORANGE, lw=2.0)pathdot,  = axR.plot([], [], "o", color=ORANGE, ms=7)axR.plot(w_fit, b_fit, "*", color="white", ms=14, mec=ORANGE_D, mew=1.3, zorder=4)axR.set_xlabel("weight w"); axR.set_ylabel("bias b")axR.set_title("loss surface  L(w, b)", loc="left")plt.tight_layout()def update(i):    wi, bi = path[i]    fitline.set_data(xs, wi*xs + bi)    title_l.set_text(f"$\\hat{{y}} = {wi:.2f}x + {bi:.2f}$")    pathline.set_data(path[:i+1, 0], path[:i+1, 1])    pathdot.set_data([wi], [bi])    return fitline, title_l, pathline, pathdotGIF_PATH = "linear_fit_gradient_descent.gif"anim = FuncAnimation(fig, update, frames=len(path), interval=90, blit=False)anim.save(GIF_PATH, writer=PillowWriter(fps=14))plt.close(fig)display(Image(filename=GIF_PATH))print(f"least-squares solution: w={w_fit:.3f}, b={b_fit:.3f}")

### Where the linear model breaksFour clusters labelled by XOR. No straight line separates them; a single hidden layer does.Both models below are trained here, so the accuracies quoted on the slide are real.

In [ ]:
def make_xor_2d(n=90, seed=7):    r = np.random.default_rng(seed)    out_X, out_Y = [], []    for cx, cy, lab in [(-1,-1,0), (1,1,0), (-1,1,1), (1,-1,1)]:        pts = r.normal([cx, cy], 0.42, size=(n//4, 2))        out_X.append(pts); out_Y.append(np.full(n//4, lab))    return np.vstack(out_X), np.concatenate(out_Y)Xx, Yx = make_xor_2d()# best possible straight line (logistic regression, full-batch gradient descent)Xb = np.hstack([Xx, np.ones((len(Xx), 1))]); wlin = np.zeros(3)for _ in range(6000):    p = 1/(1+np.exp(-Xb @ wlin))    wlin -= 0.08 * (Xb.T @ (p - Yx)) / len(Xx)acc_lin_2d = float(((1/(1+np.exp(-Xb @ wlin)) > .5).astype(int) == Yx).mean())# the same data with one hidden layerdef train_mlp_2d(X, Y, H=8, epochs=4000, lr=0.5, seed=0):    r = np.random.default_rng(seed)    W1 = r.normal(0, 1, (2, H)); b1 = np.zeros(H)    W2 = r.normal(0, 1, (H, 1)); b2 = np.zeros(1)    Yc = Y.reshape(-1, 1)    for _ in range(epochs):        A1 = np.tanh(X @ W1 + b1)        P = 1/(1+np.exp(-(A1 @ W2 + b2)))        dZ2 = (P - Yc)/len(X)        dA1 = dZ2 @ W2.T; dZ1 = dA1 * (1 - A1**2)        W2 -= lr * (A1.T @ dZ2); b2 -= lr * dZ2.sum(0)        W1 -= lr * (X.T @ dZ1);  b1 -= lr * dZ1.sum(0)    return W1, b1, W2, b2W1x, b1x, W2x, b2x = train_mlp_2d(Xx, Yx)def mlp2d(G):    return (1/(1+np.exp(-(np.tanh(G @ W1x + b1x) @ W2x + b2x)))).ravel()acc_mlp_2d = float(((mlp2d(Xx) > .5).astype(int) == Yx).mean())# extract the two decision boundaries as polylines, so PowerPoint can draw them# as editable curves rather than a bitmap contour_gx, _gy = np.meshgrid(np.linspace(-2.6, 2.6, 240), np.linspace(-2.6, 2.6, 240))_G = np.c_[_gx.ravel(), _gy.ravel()]_Zlin = (1/(1+np.exp(-(np.c_[_G, np.ones(len(_G))] @ wlin)))).reshape(_gx.shape)_Zmlp = mlp2d(_G).reshape(_gx.shape)def boundary_polylines(Z, max_pts=60):    # the 0.5 contour of Z, as a list of polylines in data coordinates    cs = plt.contour(_gx, _gy, Z, levels=[0.5])    segs = []    for seg in cs.allsegs[0]:        if len(seg) < 4:            continue        step = max(1, len(seg)//max_pts)        segs.append(np.asarray(seg)[::step].round(3).tolist())    plt.close()    return segsDATA["xor_2d"] = {    "class0": Xx[Yx == 0].round(3).tolist(),    "class1": Xx[Yx == 1].round(3).tolist(),    "line_w": [float(v) for v in wlin.round(4)],    "boundary_linear": boundary_polylines(_Zlin),    "boundary_mlp": boundary_polylines(_Zmlp),    "acc_linear": round(acc_lin_2d*100, 1),    "acc_mlp": round(acc_mlp_2d*100, 1),}gx, gy = np.meshgrid(np.linspace(-2.6, 2.6, 200), np.linspace(-2.6, 2.6, 200))fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))for a, Zg, t in [(ax[0], (1/(1+np.exp(-(np.c_[gx.ravel(), gy.ravel(),                        np.ones(gx.size)] @ wlin)))).reshape(gx.shape),                  f"linear — {acc_lin_2d*100:.0f}%"),                 (ax[1], mlp2d(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape),                  f"one hidden layer — {acc_mlp_2d*100:.0f}%")]:    a.contourf(gx, gy, Zg, levels=[0, .5, 1], colors=["#EAF1F5", "#FDECE1"])    a.contour(gx, gy, Zg, levels=[.5], colors=[ORANGE], linewidths=2.2)    a.scatter(*Xx[Yx == 0].T, color=TEAL, s=22)    a.scatter(*Xx[Yx == 1].T, color=ORANGE_D, s=26, marker="^")    a.set_title(t, loc="left"); a.set_xticks([]); a.set_yticks([])plt.tight_layout(); plt.show()print(f"linear {acc_lin_2d*100:.1f}%   one hidden layer {acc_mlp_2d*100:.1f}%")

---## 3. Activation functionsThe only hard requirement is nonlinearity: with a linear $\sigma$ the whole networkcollapses back to $\hat y = wx + b$.

In [ ]:
zs = np.linspace(-3, 3, 61)acts = {"relu": np.maximum(0, zs), "tanh": np.tanh(zs), "sigmoid": 1/(1+np.exp(-zs))}DATA["activations"] = {"z": zs.round(4).tolist(),                       **{k: v.round(4).tolist() for k, v in acts.items()}}fig, axs = plt.subplots(1, 3, figsize=(9.5, 2.5))for a, (k, v), c in zip(axs, acts.items(), [ORANGE, TEAL, PURPLE]):    a.axhline(0, color=LGRAY, lw=1); a.axvline(0, color=LGRAY, lw=1)    a.plot(zs, v, color=c, lw=2.6); a.set_title(k, loc="left"); a.set_xlabel("z")plt.tight_layout(); plt.show()

---## 4. Training the 4 → 5 → 1 networkPlain NumPy: `tanh` hidden layer, sigmoid output, cross-entropy loss, Adam optimizer.Nothing is hidden in a library — this is the entire model.

In [ ]:
def train_parity(Xtr, ytr, H=5, epochs=25000, lr=0.03, seed=0, scale=1.2,                 record_every=25):    r = np.random.default_rng(seed)    W1 = r.normal(0, scale, (4, H)); b1 = np.zeros(H)    W2 = r.normal(0, scale, (H, 1)); b2 = np.zeros(1)    W1_0, W2_0 = W1.copy(), W2.copy()          # keep the initial weights to draw    prm = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}    m = {k: np.zeros_like(v) for k, v in prm.items()}    v = {k: np.zeros_like(x) for k, x in prm.items()}    beta1, beta2, eps = 0.9, 0.999, 1e-8    Y = ytr.reshape(-1, 1); hist = []    for ep in range(epochs):        t = ep + 1        A1 = np.tanh(Xtr @ W1 + b1)        P = 1/(1+np.exp(-(A1 @ W2 + b2)))        if ep % record_every == 0:            loss = float(-np.mean(Y*np.log(P+1e-9) + (1-Y)*np.log(1-P+1e-9)))            hist.append((ep, loss))        dZ2 = (P - Y)/len(Xtr)        g = {"W2": A1.T @ dZ2, "b2": dZ2.sum(0)}        dA1 = dZ2 @ W2.T; dZ1 = dA1 * (1 - A1**2)        g["W1"] = Xtr.T @ dZ1; g["b1"] = dZ1.sum(0)        for k, arr in prm.items():            m[k] = beta1*m[k] + (1-beta1)*g[k]            v[k] = beta2*v[k] + (1-beta2)*g[k]**2            arr -= lr * (m[k]/(1-beta1**t)) / (np.sqrt(v[k]/(1-beta2**t)) + eps)    return dict(W1=W1, b1=b1, W2=W2, b2=b2, W1_0=W1_0, W2_0=W2_0,                hist=np.array(hist))def accuracy(X, y, W1, b1, W2, b2):    P = 1/(1+np.exp(-(np.tanh(X @ W1 + b1) @ W2 + b2)))    return float((((P > .5).astype(float)) == y.reshape(-1, 1)).mean())res = train_parity(Xtr, ytr, H=5, seed=0)acc0  = accuracy(Xtr, ytr, res["W1_0"], np.zeros(5), res["W2_0"], np.zeros(1))acctr = accuracy(Xtr, ytr, res["W1"], res["b1"], res["W2"], res["b2"])accte = accuracy(Xte, yte, res["W1"], res["b1"], res["W2"], res["b2"])# linear baseline on the very same taskXb2 = np.hstack([Xtr, np.ones((len(Xtr), 1))]); wl = np.zeros(5)for _ in range(20000):    p = 1/(1+np.exp(-Xb2 @ wl))    wl -= 0.05 * (Xb2.T @ (p - ytr)) / len(Xtr)Xteb = np.hstack([Xte, np.ones((len(Xte), 1))])acc_lin_parity = float(((1/(1+np.exp(-Xteb @ wl)) > .5).astype(float) == yte).mean())print(f"before training      {acc0*100:5.1f}%")print(f"after training (train){acctr*100:5.1f}%")print(f"after training (test) {accte*100:5.1f}%")print(f"linear model  (test) {acc_lin_parity*100:5.1f}%   <- cannot do XOR")

In [ ]:
DATA["parity_net"] = {    "W1_init": res["W1_0"].round(4).tolist(), "W2_init": res["W2_0"].round(4).tolist(),    "W1": res["W1"].round(4).tolist(),        "W2": res["W2"].round(4).tolist(),    "labels": ["nH", "nC", "nN", "nO"],    "acc_init": round(acc0*100, 1), "acc_train": round(acctr*100, 1),    "acc_test": round(accte*100, 1), "acc_linear": round(acc_lin_parity*100, 1),    "n_train": int(len(Xtr)), "n_test": int(len(Xte)), "maxc": MAXC,    "loss_hist": [[int(e), round(l, 4)] for e, l in res["hist"][::8]],}fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.0))ax[0].plot(res["hist"][:, 0], res["hist"][:, 1], color=TEAL, lw=2)ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")ax[0].set_title("note the plateau, then the drop", loc="left")im = ax[1].imshow(res["W1"], cmap="RdBu_r", vmin=-abs(res["W1"]).max(),                  vmax=abs(res["W1"]).max())ax[1].set_xticks(range(5)); ax[1].set_yticks(range(4))ax[1].set_yticklabels(["nH", "nC", "nN", "nO"])ax[1].set_xlabel("hidden unit"); ax[1].set_title("trained layer-1 weights", loc="left")plt.colorbar(im, ax=ax[1], shrink=.8)plt.tight_layout(); plt.show()

### Gradient descent is not guaranteed to workThe same architecture on the same data, started from different random weights. The spreadis the honest version of "training is hard": nothing changes except the initialization.

In [ ]:
SEEDS = [0, 2, 3, 5, 7, 9]seed_acc = []for sd in SEEDS:    rr = train_parity(Xtr, ytr, H=5, seed=sd)    a = accuracy(Xte, yte, rr["W1"], rr["b1"], rr["W2"], rr["b2"])    seed_acc.append(round(a*100, 1))    print(f"seed {sd}: test {a*100:5.1f}%")DATA["parity_seeds"] = {"seeds": SEEDS, "acc": seed_acc}plt.figure(figsize=(6, 2.6))cols = [ORANGE if a > 90 else (TEAL_L if a > 75 else GRAY) for a in seed_acc]plt.bar([str(s) for s in SEEDS], seed_acc, color=cols)plt.axhline(50, color=INK, ls="--"); plt.ylim(0, 105)plt.ylabel("test accuracy (%)"); plt.xlabel("random seed")plt.title("same network, different starting weights", loc="left")plt.tight_layout(); plt.show()print("range:", min(seed_acc), "-", max(seed_acc), "%")

In [ ]:
# where do the six seeds above actually end up? -- a 2-D map of the (31-D) parameter# space they are all optimizing over, now with two upgrades:#  1. real "in-between" samples: on top of denser trajectory recording, we add#     straight-line interpolations in full 31-D weight space between every pair of#     final solutions and evaluate their REAL loss -- these are genuine, evaluable#     networks, not fabricated points, and they directly test whether two solutions#     are connected by a low-loss path or separated by a barrier ("mode connectivity")#  2. a non-linear alternative to PCA: a from-scratch Isomap (k-NN graph -> geodesic#     shortest-path distances -> classical MDS), compared quantitatively against PCA#     by how well each 2-D layout preserves the TRUE 31-D distances between solutionsimport matplotlib.tri as trifrom scipy.spatial.distance import pdist, squareformfrom scipy.sparse import csr_matrixfrom scipy.sparse.csgraph import shortest_pathdef train_parity_path(Xtr, ytr, H=5, epochs=25000, lr=0.03, seed=0, scale=1.2,                      record_every=100):    """Same architecture, optimizer, and hyperparameters as train_parity() above,    but also snapshots the flattened parameter vector AND its real loss as it trains."""    r = np.random.default_rng(seed)    W1 = r.normal(0, scale, (4, H)); b1 = np.zeros(H)    W2 = r.normal(0, scale, (H, 1)); b2 = np.zeros(1)    prm = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}    m = {k: np.zeros_like(v) for k, v in prm.items()}    v = {k: np.zeros_like(v) for k, v in prm.items()}    beta1, beta2, eps = 0.9, 0.999, 1e-8    Y = ytr.reshape(-1, 1)    flat_path, loss_path = [], []    for ep in range(epochs):        t = ep + 1        A1 = np.tanh(Xtr @ W1 + b1)        P = 1/(1+np.exp(-(A1 @ W2 + b2)))        if ep % record_every == 0:            flat_path.append(np.concatenate([W1.ravel(), b1.ravel(), W2.ravel(), b2.ravel()]))            loss_path.append(float(-np.mean(Y*np.log(P+1e-9) + (1-Y)*np.log(1-P+1e-9))))        dZ2 = (P - Y)/len(Xtr)        g = {"W2": A1.T @ dZ2, "b2": dZ2.sum(0)}        dA1 = dZ2 @ W2.T; dZ1 = dA1 * (1 - A1**2)        g["W1"] = Xtr.T @ dZ1; g["b1"] = dZ1.sum(0)        for k, arr in prm.items():            m[k] = beta1*m[k] + (1-beta1)*g[k]            v[k] = beta2*v[k] + (1-beta2)*g[k]**2            arr -= lr * (m[k]/(1-beta1**t)) / (np.sqrt(v[k]/(1-beta2**t)) + eps)    flat_path.append(np.concatenate([W1.ravel(), b1.ravel(), W2.ravel(), b2.ravel()]))    A1 = np.tanh(Xtr @ W1 + b1); P = 1/(1+np.exp(-(A1 @ W2 + b2)))    loss_path.append(float(-np.mean(Y*np.log(P+1e-9) + (1-Y)*np.log(1-P+1e-9))))    return np.array(flat_path), np.array(loss_path), (W1, b1, W2, b2)def unflatten_parity(vflat, H=5):    i = 0    W1 = vflat[i:i+4*H].reshape(4, H); i += 4*H    b1 = vflat[i:i+H];                 i += H    W2 = vflat[i:i+H].reshape(H, 1);   i += H    b2 = vflat[i:i+1]    return W1, b1, W2, b2def parity_loss_at(vflat, H=5, X=Xtr, y=ytr):    W1, b1, W2, b2 = unflatten_parity(vflat, H)    A1 = np.tanh(X @ W1 + b1)    P = 1/(1+np.exp(-(A1 @ W2 + b2)))    Y = y.reshape(-1, 1)    return float(-np.mean(Y*np.log(P+1e-9) + (1-Y)*np.log(1-P+1e-9)))paths, path_losses, finals = {}, {}, {}for sd in SEEDS:    flat_path, loss_path, (W1e, b1e, W2e, b2e) = train_parity_path(Xtr, ytr, H=5, seed=sd,                                                                    record_every=100)    paths[sd] = flat_path    path_losses[sd] = loss_path    finals[sd] = accuracy(Xte, yte, W1e, b1e, W2e, b2e)# real "in-between" samples: straight-line interpolation in full 31-D weight space# between every pair of final solutions, with REAL loss evaluated along each bridgefinals_flat = {sd: paths[sd][-1] for sd in SEEDS}alphas = np.linspace(0, 1, 21)[1:-1]bridge_pts, bridge_loss = [], []for i, sdi in enumerate(SEEDS):    for sdj in SEEDS[i+1:]:        A, B = finals_flat[sdi], finals_flat[sdj]        for a in alphas:            v = (1 - a)*A + a*B            bridge_pts.append(v)            bridge_loss.append(parity_loss_at(v))bridge_pts = np.array(bridge_pts)bridge_loss = np.array(bridge_loss)print(f"{len(bridge_pts)} bridge points added; loss along these straight-line "      f"interpolations ranges {bridge_loss.min():.2f}-{bridge_loss.max():.2f} "      f"(vs. {min(l[-1] for l in path_losses.values()):.2f}-"      f"{max(l[-1] for l in path_losses.values()):.2f} at the six real endpoints) "      f"-- straight lines between solutions cross real loss barriers.")# pool every real point together, keeping track of where each seed's own trajectory# sits within the pool so it can still be drawn as a connected line afterwardseed_ranges = {}pool_pts, pool_loss = [], []for sd in SEEDS:    start = len(pool_pts)    pool_pts.extend(paths[sd]); pool_loss.extend(path_losses[sd])    seed_ranges[sd] = (start, len(pool_pts))pool_pts.extend(bridge_pts); pool_loss.extend(bridge_loss)pool_pts = np.array(pool_pts); pool_loss = np.array(pool_loss)n = len(pool_pts)print(f"{n} total real samples in the pool "      f"({sum(len(path_losses[sd]) for sd in SEEDS)} trajectory points + "      f"{len(bridge_pts)} bridge points)")# ---------- embedding 1: PCA (linear) ----------mean_pt = pool_pts.mean(0)_, S, Vt = np.linalg.svd(pool_pts - mean_pt, full_matrices=False)pca_xy = (pool_pts - mean_pt) @ Vt[:2].Tpct_var = 100 * S**2 / np.sum(S**2)# ---------- embedding 2: Isomap (non-linear), from scratch ----------# 1) k-nearest-neighbour graph on TRUE 31-D Euclidean distances# 2) shortest-path (geodesic) distance over that graph -- distance along the#    (possibly curved) manifold the solutions live on, instead of a straight line# 3) classical MDS on the geodesic distances -> 2-D coordinatesD_full = squareform(pdist(pool_pts))K = 15knn = np.argsort(D_full, axis=1)[:, 1:K+1]rows = np.repeat(np.arange(n), K); cols = knn.ravel(); vals = D_full[rows, cols]graph = csr_matrix((np.concatenate([vals, vals]),                    (np.concatenate([rows, cols]), np.concatenate([cols, rows]))), shape=(n, n))geo = shortest_path(graph, method="D", directed=False)if np.isinf(geo).any():    geo[np.isinf(geo)] = geo[np.isfinite(geo)].max() * 1.5Jm = np.eye(n) - np.ones((n, n)) / nBm = -0.5 * Jm @ geo**2 @ Jmeigval, eigvec = np.linalg.eigh(Bm)order = np.argsort(eigval)[::-1]iso_xy = eigvec[:, order][:, :2] * np.sqrt(np.maximum(eigval[order][:2], 0))# how well does each 2-D layout preserve the TRUE 31-D geometry? sample random pairs# and correlate their true distance with their 2-D distancerng = np.random.default_rng(0)pi, pj = rng.integers(0, n, 6000), rng.integers(0, n, 6000)keep = pi != pj; pi, pj = pi[keep], pj[keep]d_true = D_full[pi, pj]r_pca = np.corrcoef(d_true, np.linalg.norm(pca_xy[pi]-pca_xy[pj], axis=1))[0, 1]r_iso = np.corrcoef(d_true, np.linalg.norm(iso_xy[pi]-iso_xy[pj], axis=1))[0, 1]print(f"\nhow well does each 2-D map preserve true 31-D distances between solutions?")print(f"  PCA    (linear):     r = {r_pca:.3f}")print(f"  Isomap (non-linear): r = {r_iso:.3f}  (k={K} nearest neighbours)")better = "PCA" if r_pca > r_iso else "Isomap"print(f"  -> {better} preserves the true geometry better for this data.")def plot_landscape(ax, xy, loss, title, xlabel, ylabel):    triang = tri.Triangulation(xy[:, 0], xy[:, 1])    xt, yt = xy[triang.triangles, 0], xy[triang.triangles, 1]    edges = [np.hypot(xt[:, i]-xt[:, j], yt[:, i]-yt[:, j]) for i, j in [(0, 1), (1, 2), (2, 0)]]    max_edge = np.max(edges, axis=0)    triang.set_mask(max_edge > 3 * np.median(max_edge))    cf = ax.tricontourf(triang, loss, levels=24, cmap="Blues_r")    for sd in SEEDS:        s0, s1 = seed_ranges[sd]        p2 = xy[s0:s1]; acc = finals[sd] * 100        col = ORANGE if acc > 90 else (TEAL_L if acc > 75 else GRAY)        ax.plot(p2[:, 0], p2[:, 1], color=col, lw=1.6, alpha=0.9)        ax.plot(p2[0, 0], p2[0, 1], "o", color=col, ms=5, mec="white", mew=0.7, zorder=4)        ax.plot(p2[-1, 0], p2[-1, 1], "*", color=col, ms=12, mec="white", mew=0.7, zorder=4)    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)    ax.set_title(title, loc="left", fontsize=11)    return cffig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5.6))cf1 = plot_landscape(axL, pca_xy, pool_loss, f"PCA  (r={r_pca:.2f} vs. true distances)",                     f"PC1 ({pct_var[0]:.1f}% var)", f"PC2 ({pct_var[1]:.1f}% var)")cf2 = plot_landscape(axR, iso_xy, pool_loss, f"Isomap, k={K}  (r={r_iso:.2f} vs. true distances)",                     "Isomap dim 1", "Isomap dim 2")fig.colorbar(cf1, ax=axL, shrink=0.85, label="training loss (BCE)")fig.colorbar(cf2, ax=axR, shrink=0.85, label="training loss (BCE)")plt.tight_layout(); plt.show()print("\nWhy doesn't the non-linear map win here? Isomap's advantage over PCA is "      "unfolding data that truly lies on a CURVED manifold. Our data is the opposite: six "      "fairly direct optimization paths plus literal straight-line bridges between them -- "      "there is little curvature to unfold, and with only 6 trajectories the k-NN graph is "      "sparsely connected BETWEEN them, so geodesic distances between different seeds get "      "inflated relative to their true straight-line distance. A linear projection has "      "nothing to gain from detouring through a graph here, so it wins.")DATA["parity_seed_landscape"] = {    "pca_xy": pca_xy.round(4).tolist(), "iso_xy": iso_xy.round(4).tolist(),    "loss": pool_loss.round(4).tolist(),    "seed_ranges": {str(sd): list(seed_ranges[sd]) for sd in SEEDS},    "final_acc": {str(sd): round(finals[sd] * 100, 1) for sd in SEEDS},    "pct_var": [round(float(pct_var[0]), 1), round(float(pct_var[1]), 1)],    "r_pca": round(float(r_pca), 3), "r_iso": round(float(r_iso), 3),}# ---- additional figure: the final layer-1 (input -> hidden) weights, one per seed ----# same rows (nH, nC, nN, nO) as the single-seed heatmap earlier in the notebook, but# now compared side by side across all six solutions on a shared color scalefinals_W1 = {sd: unflatten_parity(paths[sd][-1])[0] for sd in SEEDS}vmax = max(np.abs(w).max() for w in finals_W1.values())fig2, axs = plt.subplots(1, len(SEEDS), figsize=(13.5, 3.0))for ax, sd in zip(axs, SEEDS):    im = ax.imshow(finals_W1[sd], cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")    ax.set_xticks(range(5)); ax.set_yticks(range(4))    ax.set_yticklabels(["nH", "nC", "nN", "nO"] if sd == SEEDS[0] else [])    ax.set_xlabel("hidden unit")    ax.set_title(f"seed {sd} ({finals[sd]*100:.0f}%)", fontsize=10)fig2.suptitle("trained layer-1 weights, one solution per seed", x=0.02, ha="left", fontsize=11)fig2.colorbar(im, ax=axs, shrink=0.85, label="weight value", pad=0.015)plt.show()for sd in SEEDS:    print(f"seed {sd}: test acc {finals[sd]*100:5.1f}%   final train loss {path_losses[sd][-1]:.4f}")print(f"\nPC1 explains {pct_var[0]:.1f}% of parameter-space variance, "      f"PC2 explains {pct_var[1]:.1f}% ({pct_var[0]+pct_var[1]:.1f}% combined).")# ---- why do all six starting points look so close together? ----init_pts = np.array([paths[sd][0] for sd in SEEDS])final_pts = np.array([paths[sd][-1] for sd in SEEDS])def _pairwise_spread(pts):    d = pts[:, None, :] - pts[None, :, :]    iu = np.triu_indices(len(pts), k=1)    return np.sqrt((d**2).sum(-1))[iu]init_spread  = _pairwise_spread(init_pts).mean()final_spread = _pairwise_spread(final_pts).mean()print(f"\nmean pairwise distance between seeds, in the full 31-D parameter space:")print(f"  at initialization: {init_spread:6.1f}")print(f"  after training:    {final_spread:6.1f}   ({final_spread/init_spread:.0f}x larger)")print("All six seeds draw W1, W2 from the same N(0, 1.2) distribution, so they start")print("a fairly ordinary random distance apart -- but training pushes the tanh units")print("toward large, confident, saturated activations to fit this sharp parity boundary,")print("growing the weight norms roughly 10-20x and scattering the six solutions far")print("apart in very different directions. Once the SAME PCA axes (fit on everything,")print("start and end points alike) have to stretch to cover that huge final spread, the")print("comparatively small initial differences between seeds all collapse visually into")print("a tight little cluster near the plot's center -- it's a difference of scale, not")print("because the six initializations were actually close to identical.")

---## 5. The loss surface and the learning rateA two-weight cartoon, because that is all we can draw. The deck renders this as aneditable contour schematic rather than a bitmap.

In [ ]:
# a genuine 2-parameter neural network: yhat(x) = w2 * tanh(w1 * x)# (one input weight, one output weight, no biases) fit to noisy data by gradient# descent on its REAL loss surface -- animated, with the fitted curve updating# in lockstep with the weights.## Same starting point, same data, same number of steps -- only the learning rate# eta differs, so any difference you see below is caused by eta alone:#   too slow  -- barely moves in the time available#   good      -- converges cleanly#   too fast  -- overshoots and bounces around instead of settlingxd = np.linspace(-1, 1, 24)ftrue2 = lambda t: np.sin(2.2*t)yd = ftrue2(xd) + np.random.default_rng(11).normal(0, 0.12, len(xd))def nn_pred(w1, w2, x):    return w2 * np.tanh(w1 * x)def nn_loss(w1, w2, x=xd, y=yd):    return np.mean((nn_pred(w1, w2, x) - y)**2)def nn_grad(w1, w2, x=xd, y=yd):    """Analytic gradient of the MSE loss w.r.t. (w1, w2) -- backprop by hand."""    t = np.tanh(w1 * x)    err = w2*t - y    dW1 = np.mean(2*err*w2*(1 - t**2)*x)    dW2 = np.mean(2*err*t)    return np.array([dW1, dW2])w0, n_steps = np.array([1.5, -1.0]), 80REGIMES = {"too slow": 0.02, "good": 0.4, "too fast": 1.9}paths = {}for name, lr in REGIMES.items():    w = w0.copy(); path = [w.copy()]    for _ in range(n_steps):        w = w - lr*nn_grad(*w)        path.append(w.copy())    paths[name] = np.array(path)# keep feeding the PPTX pipeline the same shape of data as before (single path)# -- fig_loss_contour() in nativefigs.py only ever needs one trajectory, so it# gets the well-tuned "good" run; the full three-way comparison is saved# alongside it under its own key for anyone who wants to build a slide from itDATA["loss_surface"] = {"path": paths["good"].round(4).tolist(),                        "levels": [round(float(nn_loss(*q)), 4) for q in paths["good"]]}DATA["learning_rate_2d"] = {    name: {"lr": lr, "path": paths[name].round(4).tolist(),           "final_loss": round(float(nn_loss(*paths[name][-1])), 4)}    for name, lr in REGIMES.items()}g1, g2 = np.meshgrid(np.linspace(-3, 4, 140), np.linspace(-3, 3, 140))Z = np.array([[nn_loss(a, b) for a in g1[0]] for b in g2[:, 0]])xq2 = np.linspace(-1.05, 1.05, 80)from matplotlib.animation import FuncAnimation, PillowWriterfrom IPython.display import Image, displayfig, axs = plt.subplots(2, 3, figsize=(11, 6.2))artists = {}for j, (name, lr) in enumerate(REGIMES.items()):    axL, axR = axs[0, j], axs[1, j]    axL.contourf(g1, g2, Z, levels=24, cmap="Blues_r")    pl, = axL.plot([], [], color=ORANGE, lw=2.0)    pd, = axL.plot([], [], "o", color=ORANGE, ms=6)    axL.set_title(f"{name}  ($\\eta$={lr})", loc="left", fontsize=11)    axL.set_xticks([]); axL.set_yticks([])    axR.scatter(xd, yd, color=TEAL, s=18, zorder=3)    axR.plot(xq2, ftrue2(xq2), color=LGRAY, ls="--", lw=1.5)    fl, = axR.plot([], [], color=ORANGE, lw=2.2)    axR.set_ylim(-1.6, 1.6); axR.set_xticks([]); axR.set_yticks([])    artists[name] = (pl, pd, fl)plt.tight_layout()def update(i):    out = []    for name in REGIMES:        path = paths[name]        pl, pd, fl = artists[name]        w1i, w2i = path[i]        pl.set_data(path[:i+1, 0], path[:i+1, 1])        pd.set_data([w1i], [w2i])        fl.set_data(xq2, nn_pred(w1i, w2i, xq2))        out += [pl, pd, fl]    return outGIF_PATH = "loss_surface_lr_comparison.gif"anim = FuncAnimation(fig, update, frames=n_steps+1, interval=90, blit=False)anim.save(GIF_PATH, writer=PillowWriter(fps=14))plt.close(fig)display(Image(filename=GIF_PATH))for name, lr in REGIMES.items():    print(f"{name:9s} eta={lr:<5} final loss={nn_loss(*paths[name][-1]):.4f}")

In [ ]:
# learning rate: three regimes on a 1-D parabolaf  = lambda w: (w-1.0)**2 + 0.6df = lambda w: 2*(w-1.0)lr_data = {}for name, lr in [("small", 0.035), ("good", 0.35), ("large", 1.02)]:    w, pts = -2.2, [-2.2]    for _ in range(11):        w = w - lr*df(w); pts.append(w)    lr_data[name] = {"lr": lr, "w": [round(float(v), 4) for v in pts],                     "loss": [round(float(f(v)), 4) for v in pts]}wq = np.linspace(-2.6, 4.6, 60)DATA["learning_rate"] = {"curve_w": wq.round(3).tolist(),                         "curve_loss": f(wq).round(3).tolist(), **lr_data}fig, axs = plt.subplots(1, 3, figsize=(9.5, 2.4))for a, (name, d) in zip(axs, lr_data.items()):    a.plot(wq, f(wq), color=TEAL, lw=2)    a.plot(d["w"], d["loss"], color=ORANGE, marker="o", ms=4, lw=1.4)    a.set_title(f"$\\eta$ {name} = {d['lr']}", loc="left")    a.set_ylim(0, 14); a.set_xlabel("w")plt.tight_layout(); plt.show()

---## 6. Overfitting and regularization

In [ ]:
# small neural nets of increasing capacity on the same 14 points -- the NN analogue# of "increasing polynomial degree": here capacity is controlled by hidden-layer# width H, with every other hyperparameter written out explicitlydef train_mlp_1d(x, y, H, epochs, lr, seed=0, scale=1.0):    """One-hidden-layer tanh network, scalar regression output, trained with Adam."""    r = np.random.default_rng(seed)    X = x.reshape(-1, 1); Y = y.reshape(-1, 1)    W1 = r.normal(0, scale, (1, H)); b1 = np.zeros(H)    W2 = r.normal(0, scale, (H, 1)); b2 = np.zeros(1)    prm = {"W1": W1, "b1": b1, "W2": W2, "b2": b2}    m = {k: np.zeros_like(v) for k, v in prm.items()}    v = {k: np.zeros_like(v) for k, v in prm.items()}    beta1, beta2, eps = 0.9, 0.999, 1e-8    for ep in range(epochs):        t = ep + 1        A1 = np.tanh(X @ W1 + b1)        pred = A1 @ W2 + b2        dZ2 = (pred - Y) / len(X)        g = {"W2": A1.T @ dZ2, "b2": dZ2.sum(0)}        dA1 = dZ2 @ W2.T; dZ1 = dA1 * (1 - A1**2)        g["W1"] = X.T @ dZ1; g["b1"] = dZ1.sum(0)        for k, arr in prm.items():            m[k] = beta1*m[k] + (1-beta1)*g[k]            v[k] = beta2*v[k] + (1-beta2)*g[k]**2            arr -= lr * (m[k]/(1-beta1**t)) / (np.sqrt(v[k]/(1-beta2**t)) + eps)    return W1, b1, W2, b2def predict_mlp_1d(xq, W1, b1, W2, b2):    A1 = np.tanh(xq.reshape(-1, 1) @ W1 + b1)    return (A1 @ W2 + b2).ravel()np_rng = np.random.default_rng(3)xp = np.sort(np_rng.uniform(-1, 1, 14))ftrue = lambda t: np.sin(2.4*t) + 0.35*typ = ftrue(xp) + np_rng.normal(0, 0.18, 14)xq = np.linspace(-1.05, 1.05, 60)# same data, same optimizer and seed throughout -- only capacity (H) and how long# each net is trained differ between panels, exactly like the parameters you'd# report for any trained modelCAPACITY = {    "h1":  dict(H=1,  epochs=8000,  lr=0.03, seed=0, label="1 hidden unit"),    "h4":  dict(H=4,  epochs=1000,  lr=0.03, seed=0, label="4 hidden units"),    "h64": dict(H=64, epochs=40000, lr=0.03, seed=0, label="64 hidden units"),}fits = {}for key, cfg in CAPACITY.items():    W1, b1, W2, b2 = train_mlp_1d(xp, yp, cfg["H"], cfg["epochs"], cfg["lr"], cfg["seed"])    fits[key] = np.clip(predict_mlp_1d(xq, W1, b1, W2, b2), -2.5, 2.5).round(4).tolist()DATA["overfit_capacity"] = {    "x": xp.round(4).tolist(), "y": yp.round(4).tolist(),    "xq": xq.round(4).tolist(), "truth": ftrue(xq).round(4).tolist(),    **fits, "labels": {k: v["label"] for k, v in CAPACITY.items()},}fig, axs = plt.subplots(1, 3, figsize=(9.5, 2.5))for a, key in zip(axs, CAPACITY):    a.plot(xq, ftrue(xq), color=LGRAY, ls="--", lw=2)    a.plot(xq, fits[key], color=ORANGE, lw=2.2)    a.scatter(xp, yp, color=TEAL, s=25)    a.set_ylim(-2.4, 2.4); a.set_title(CAPACITY[key]["label"], loc="left")plt.tight_layout(); plt.show()for key, cfg in CAPACITY.items():    print(f"{cfg['label']:15s} H={cfg['H']:3d} epochs={cfg['epochs']:6d}  "          f"train MSE={np.mean((predict_mlp_1d(xp, *train_mlp_1d(xp, yp, cfg['H'], cfg['epochs'], cfg['lr'], cfg['seed'])) - yp)**2):.4f}")

In [ ]:
# train / validation curves, and the early-stopping pointep = np.arange(1, 121)tr_loss = 1.5*np.exp(-ep/26) + 0.05va_loss = 1.5*np.exp(-ep/22) + 0.19 + 0.0055*np.clip(ep-42, 0, None)best = int(np.argmin(va_loss))DATA["overfit_curves"] = {    "epoch": ep[::3].tolist(),    "train": tr_loss[::3].round(4).tolist(),    "val":   va_loss[::3].round(4).tolist(),    "best_epoch": int(ep[best]), "best_val": round(float(va_loss[best]), 4),}# weight histograms with and without L2r4 = np.random.default_rng(4)BINS = np.linspace(-4.5, 4.5, 16)          # few enough bins to label on a slideh_no, _ = np.histogram(r4.normal(0, 1.35, 4000), bins=BINS)h_wd, _ = np.histogram(r4.normal(0, 0.42, 4000), bins=BINS)centers = (BINS[:-1] + BINS[1:]) / 2DATA["weight_decay"] = {"centers": centers.round(3).tolist(),                        "no_penalty": h_no.tolist(), "with_decay": h_wd.tolist()}fig, axs = plt.subplots(1, 2, figsize=(9, 2.7))axs[0].plot(ep, tr_loss, color=TEAL, lw=2, label="train")axs[0].plot(ep, va_loss, color=ORANGE, lw=2, label="validation")axs[0].axvline(ep[best], color=INK, ls="--")axs[0].legend(); axs[0].set_xlabel("epoch"); axs[0].set_ylabel("loss")axs[0].set_title(f"stop at epoch {ep[best]}", loc="left")axs[1].bar(centers, h_no, width=.34, color=LGRAY, label="no penalty")axs[1].bar(centers, h_wd, width=.34, color=ORANGE, label="weight decay")axs[1].legend(); axs[1].set_xlabel("weight value")axs[1].set_title("L2 shrinks the weights", loc="left")plt.tight_layout(); plt.show()print("early stopping epoch:", ep[best])

---## 7. SchematicsThese are pure geometry — the deck draws them as native PowerPoint shapes, so only thelayout constants travel in the JSON.

In [ ]:
DATA["timeline"] = [    {"year": "1958", "head": "Perceptron",       "sub": "Rosenblatt builds the|first trainable neuron"},    {"year": "1986", "head": "Backpropagation",  "sub": "Rumelhart, Hinton|& Williams"},    {"year": "2012", "head": "AlexNet",          "sub": "15.3% vs 26.2% error|on ImageNet"},    {"year": "2017", "head": "MPNNs",            "sub": "message passing for|quantum chemistry"},    {"year": "2021", "head": "AlphaFold2",       "sub": "~200M protein|structures released"},    {"year": "2023", "head": "GNoME",            "sub": "380k+ new stable|crystals predicted"},    {"year": "2024", "head": "Two Nobel Prizes", "sub": "Physics: neural networks|Chemistry: protein design"},]# max-pooling worked examplepool_in = [[1,3,2,4],[5,6,1,2],[7,8,3,1],[4,2,9,5]]pool_out = [[max(pool_in[i][j], pool_in[i][j+1], pool_in[i+1][j], pool_in[i+1][j+1])             for j in (0,2)] for i in (0,2)]DATA["pooling"] = {"input": pool_in, "output": pool_out}print("max-pool 2x2:", pool_out)# parameter count for the network on the slidesD_in, H, D_out = 4, 5, 1DATA["param_count"] = {    "rows": [["layer 1 weights", f"{D_in} x {H}", D_in*H],             ["layer 1 biases",  f"{H}",          H],             ["layer 2 weights", f"{H} x {D_out}", H*D_out],             ["layer 2 bias",    f"{D_out}",      D_out]],    "total": D_in*H + H + H*D_out + D_out,}print("total trainable parameters:", DATA["param_count"]["total"])

---## 8. ExportWrites `figure_data.json`. Then run:```bashpython build_deck.py```which produces `lecture_04-intro-deep-learning.pptx` with every figure as a **native,editable** PowerPoint chart, shape or table.

In [ ]:
with open("figure_data.json", "w") as fh:    json.dump(DATA, fh, indent=1)print("wrote figure_data.json\n")print("Numbers quoted on the slides")print("-" * 44)pn = DATA["parity_net"]print(f"  linear model, 2-D XOR      {DATA['xor_2d']['acc_linear']:>6}%")print(f"  one hidden layer, 2-D XOR  {DATA['xor_2d']['acc_mlp']:>6}%")print(f"  network before training    {pn['acc_init']:>6}%")print(f"  network after training     {pn['acc_test']:>6}%  (test)")print(f"  linear model, parity task  {pn['acc_linear']:>6}%")print(f"  seed spread                {min(DATA['parity_seeds']['acc'])}-"      f"{max(DATA['parity_seeds']['acc'])}%")print(f"  trainable parameters       {DATA['param_count']['total']:>6}")print(f"  early stopping epoch       {DATA['overfit_curves']['best_epoch']:>6}")